# Centred point cells and Lloyd relaxation

[Issue #31](https://github.com/rhugman/vorflow/issues/31) reports that point
features (wells, observation points) do not always end up in cells that are
centred on, and symmetric around, the point, and asks whether Lloyd iterations
are available. A vorflow cell's generator is a Gmsh triangle-mesh node, so the
point is always the generator of its own cell, but nothing makes that generator
the cell's centroid, and the cell's shape depends on where Gmsh puts the
neighbouring nodes.

This notebook measures the problem on a model with four refined wells and
compares three options:

1. Gmsh smoothing (`MeshGenerator(smoothing_steps=..., optimization_cycles=...)`),
   which already exists and does not fix it;
2. `ConceptualMesh.add_point(..., hex_ring=True)`, which makes the point's own
   cell a regular hexagon centred on the point;
3. `VoronoiTessellator(..., lloyd_iterations=N)`, a density-weighted Lloyd
   relaxation of the generators across the whole grid.

Both new options are off by default. The last section combines them and gives
guidance on when to use which. The notebook runs in about 30 s.

In [ ]:
import copy
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely
from shapely.geometry import LineString, Point, Polygon, box

from vorflow import ConceptualMesh, MeshGenerator, VoronoiTessellator, set_verbosity
from vorflow.utils import build_connectivity, calculate_mesh_quality

set_verbosity(0)
pd.set_option("display.precision", 3)

## 1. The problem

A 2 km x 2 km domain (zone 1) with an upper zone (zone 2) whose lower edge is
irregular, and four wells refined to 5, 10, 7 and 5 m with the default growth
factor of 1.2. Away from the features the mesh size is `background_lc = 100` m.

Metrics used throughout:

- `drift_ratio` (from `vorflow.utils.calculate_mesh_quality`): distance from a
  cell's generator to its centroid divided by sqrt(cell area). 0 means the
  generator is the centroid.
- Neighbour-area CV: coefficient of variation of the areas of the cells that
  touch a well's cell. 0 means the ring of cells around the well is symmetric
  in size.
- `ortho_error` (from `vorflow.utils.build_connectivity(grid, center="centroid")`):
  angle in degrees between the centroid-to-centroid connector and the normal of
  the shared face. MODFLOW 6 places cell centres at the centroids it is given,
  so this is the orthogonality error the flow solution sees.
- "Interior" cells are the cells that do not touch the domain boundary; the
  boundary cells are clipped and have their own large drift.

In [ ]:
DOMAIN = box(0.0, 0.0, 2000.0, 2000.0)
# Upper zone with an irregular lower edge.
UPPER_ZONE = Polygon([
    (0, 1150), (300, 1080), (520, 1210), (800, 1120), (1050, 1260),
    (1300, 1150), (1600, 1230), (1800, 1100), (2000, 1180), (2000, 2000), (0, 2000),
])
# well id: (location, resolution in m)
WELLS = {
    "w0": (Point(420.0, 520.0), 5.0),
    "w1": (Point(1450.0, 700.0), 10.0),
    "w2": (Point(700.0, 1600.0), 7.0),
    "w3": (Point(1500.0, 1650.0), 5.0),
}
BACKGROUND_LC = 100.0
GROWTH_FACTOR = 1.2

In [ ]:
def build_conceptual_mesh(hex_ring: bool = False, domain=DOMAIN, zone=UPPER_ZONE, wells=WELLS,
                          growth_factor=GROWTH_FACTOR) -> ConceptualMesh:
    """Domain (zone 1), upper zone (zone 2) and the wells."""
    cm = ConceptualMesh()
    cm.add_polygon(domain, zone_id=1)
    cm.add_polygon(zone, zone_id=2, z_order=1)
    for well_id, (point, resolution) in wells.items():
        cm.add_point(point, point_id=well_id, resolution=resolution,
                     growth_factor=growth_factor, hex_ring=hex_ring)
    return cm


def mesh_case(hex_ring: bool = False, mg_kwargs: dict | None = None, background_lc: float = BACKGROUND_LC):
    """(conceptual mesh, mesh generator, seconds spent meshing) for one case."""
    cm = build_conceptual_mesh(hex_ring)
    clean = cm.generate()
    start = time.perf_counter()
    mg = MeshGenerator(background_lc=background_lc, verbosity=0, **(mg_kwargs or {}))
    mg.generate(*clean)
    return cm, mg, time.perf_counter() - start


def tessellate(cm: ConceptualMesh, mg: MeshGenerator, lloyd_iterations: int = 0):
    """(grid, tessellator, seconds spent tessellating)."""
    start = time.perf_counter()
    tess = VoronoiTessellator(mg, cm, lloyd_iterations=lloyd_iterations)
    grid = tess.generate()
    return grid, tess, time.perf_counter() - start


def well_cell_table(grid, wells=WELLS) -> pd.DataFrame:
    """drift_ratio, neighbour-area CV, area and neighbour count of the cell holding each well."""
    quality = calculate_mesh_quality(grid)
    geoms = quality.geometry.to_numpy()
    tree = shapely.STRtree(geoms)
    rows = []
    for well_id, (point, resolution) in wells.items():
        pos = int(np.flatnonzero(shapely.contains(geoms, point))[0])
        neighbours = [n for n in tree.query(geoms[pos], predicate="touches") if n != pos]
        areas = shapely.area(geoms[neighbours])
        rows.append({
            "well": well_id,
            "resolution": resolution,
            "drift_ratio": quality["drift_ratio"].iloc[pos],
            "nbr_area_cv": np.std(areas) / np.mean(areas),
            "n_neighbours": len(neighbours),
            "cell_area": geoms[pos].area,
        })
    return pd.DataFrame(rows).set_index("well")


def short_face_share(geoms, ratio: float = 0.1) -> float:
    """Share of cells with an edge shorter than ``ratio`` x sqrt(cell area)."""
    shortest = np.array([np.min(np.hypot(*np.diff(np.asarray(g.exterior.coords), axis=0).T)) for g in geoms])
    return float(np.mean(shortest < ratio * np.sqrt(shapely.area(geoms))))


def summarise(name: str, grid, runtime: float, domain=DOMAIN, wells=WELLS) -> dict:
    """One row of grid-wide and well-cell metrics."""
    quality = calculate_mesh_quality(grid)
    interior = ~quality.geometry.intersects(domain.boundary)
    drift = quality.loc[interior, "drift_ratio"]
    ortho = build_connectivity(grid, center="centroid")["ortho_error"]
    well = well_cell_table(grid, wells)
    return {
        "case": name,
        "n_cells": len(grid),
        "runtime_s": runtime,
        "well_drift": well["drift_ratio"].mean(),
        "well_nbr_area_cv": well["nbr_area_cv"].mean(),
        "int_drift_med": drift.median(),
        "int_drift_p95": drift.quantile(0.95),
        "ortho_p95_centroid": ortho.quantile(0.95),
        "well_cell_area": well["cell_area"].mean(),
        "short_face_cells": short_face_share(quality.geometry.to_numpy()[interior]),
    }

In [ ]:
def plot_overview(grid, domain=DOMAIN, zone=UPPER_ZONE, wells=WELLS):
    """Whole grid coloured by zone, with the zone edge and the wells."""
    fig, ax = plt.subplots(figsize=(7, 7))
    grid.plot(ax=ax, column="zone_id", cmap="Pastel1", edgecolor="0.4", linewidth=0.2)
    ax.plot(*zone.exterior.xy, color="tab:blue", lw=1, label="upper zone edge")
    for well_id, (point, _) in wells.items():
        ax.plot(point.x, point.y, "r*", ms=10)
        ax.annotate(well_id, (point.x, point.y), xytext=(6, 6), textcoords="offset points", color="r")
    ax.plot(*domain.exterior.xy, color="k", lw=1)
    ax.set_aspect("equal")
    ax.set_title(f"Baseline grid: {len(grid)} cells, zones 1 and 2, wells as red stars")
    ax.legend(loc="lower right")
    fig.tight_layout()
    return fig


def plot_well_zooms(grid, title: str, wells=WELLS, half_width_cells: float = 4.0):
    """One panel per well: cell edges, generators, centroids and the well's own cell."""
    fig, axes = plt.subplots(1, len(wells), figsize=(3.6 * len(wells), 4.0), layout="constrained")
    for ax, (well_id, (point, resolution)) in zip(axes, wells.items()):
        half = half_width_cells * resolution
        window = box(point.x - half, point.y - half, point.x + half, point.y + half)
        local = grid[grid.intersects(window)]
        local[local.contains(point)].plot(ax=ax, color="tab:orange", alpha=0.45)
        local.boundary.plot(ax=ax, color="0.3", lw=0.7)
        centroids = local.geometry.centroid
        ax.plot(local["x"], local["y"], "k.", ms=4, label="generator")
        ax.plot(centroids.x, centroids.y, "+", color="tab:blue", ms=7, mew=1.2, label="centroid")
        ax.plot(point.x, point.y, "r*", ms=11, label="well")
        ax.set_xlim(point.x - half, point.x + half)
        ax.set_ylim(point.y - half, point.y + half)
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f"{well_id}, resolution {resolution:g} m")
    axes[0].legend(loc="lower left", fontsize=7, framealpha=0.9)
    fig.suptitle(title)
    return fig

In [ ]:
cm_base, mg_base, mesh_s = mesh_case()
grid_base, _, tess_s = tessellate(cm_base, mg_base)
print(f"{len(grid_base)} cells, meshing {mesh_s:.2f} s, tessellating {tess_s:.2f} s")
plot_overview(grid_base);

In [ ]:
plot_well_zooms(grid_base, "Defaults: the well (red) is its cell's generator but not its centroid (blue +)");
well_base = well_cell_table(grid_base)
well_base.round(3)

Each well sits on its cell's generator, but the cell (orange) is an irregular
polygon with 5 to 7 neighbours, the well is 4-10% of the cell width
(`drift_ratio` 0.04-0.10) off the centroid, and the neighbour-area CV is
0.15-0.23. The shape depends on where Gmsh's frontal-Delaunay mesher
happened to put the first ring of triangle nodes around the embedded point.

## 2. Why Gmsh smoothing does not fix it

`MeshGenerator` exposes two Gmsh options: `smoothing_steps` (Gmsh
`Mesh.Smoothing`, default 10) and `optimization_cycles` (extra `Relocate2D` +
`Laplace2D` passes, default 2). Both are Laplacian-type smoothing of the
*triangle* mesh: each free node moves towards the average of its neighbouring
nodes, which improves the triangle shapes. The Voronoi cell of a node is a
different object. Its centroid is the area-weighted mean of the cell, not the
mean of the neighbouring nodes, and on a graded mesh the two differ
systematically. The well itself is an embedded node that smoothing never moves,
so its cell shape is decided only by its neighbours.

The sweep below compares no smoothing, the default (10/2) and 10x the default (100/10).

In [ ]:
SMOOTHING_CASES = {
    "smoothing 0/0": {"smoothing_steps": 0, "optimization_cycles": 0},
    "smoothing 10/2 (default)": {},
    "smoothing 100/10": {"smoothing_steps": 100, "optimization_cycles": 10},
}
rows = []
for name, mg_kwargs in SMOOTHING_CASES.items():
    cm, mg, mesh_s = mesh_case(mg_kwargs=mg_kwargs)
    grid, _, tess_s = tessellate(cm, mg)
    rows.append(summarise(name, grid, mesh_s + tess_s))
smoothing_table = pd.DataFrame(rows).set_index("case")
smoothing_table[["runtime_s", "well_drift", "well_nbr_area_cv", "int_drift_med", "int_drift_p95", "ortho_p95_centroid"]]

Going from no smoothing to the default lowers the interior p95 drift a little
and the p95 centroid `ortho_error` by about 1.5 degrees. Ten times more
smoothing changes neither, costs about five times the runtime, and leaves the
well cells as they were. Smoothing is the right tool for triangle quality, not
for centring Voronoi generators.

## 3. `hex_ring=True`: a regular hexagon around the point

`add_point(..., hex_ring=True)` adds six fixed mesh nodes ("seeds") at radius
r = `resolution` around the point, at 30 + k x 60 degrees. The point and two
adjacent seeds form an equilateral triangle with side r, so the six triangles
around the point are equilateral. The point's Voronoi cell is then bounded by
the six perpendicular bisectors at distance r/2: a regular hexagon centred on
the point with apothem r/2, vertices at r/sqrt(3) and area (sqrt(3)/2) r^2.
Nothing else in the mesh can cut it as long as no other node comes inside the
ring, which is why the ring is dropped near other features (below).

In [ ]:
def plot_hex_ring_diagram(r: float = 1.0):
    """Point, six seeds at radius r, the equilateral triangles and the resulting hexagon."""
    seed_angles = np.radians(30.0 + 60.0 * np.arange(6))
    seeds = r * np.column_stack([np.cos(seed_angles), np.sin(seed_angles)])
    vertex_angles = np.radians(60.0 * np.arange(6))
    hexagon = r / np.sqrt(3.0) * np.column_stack([np.cos(vertex_angles), np.sin(vertex_angles)])

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.fill(*hexagon.T, color="tab:orange", alpha=0.45, label="point's Voronoi cell")
    for a, b in zip(seeds, np.roll(seeds, -1, axis=0)):
        ax.plot([0, a[0], b[0], 0], [0, a[1], b[1], 0], color="0.6", lw=0.8)
    ax.add_patch(plt.Circle((0, 0), r, fill=False, ls="--", color="0.5"))
    ax.plot(*seeds.T, "ko", ms=6, label="ring seeds (fixed nodes)")
    ax.plot(0, 0, "r*", ms=14, label="point")
    ax.annotate("", xy=seeds[1], xytext=(0, 0), arrowprops={"arrowstyle": "<->", "color": "tab:blue"})
    ax.text(*(0.55 * seeds[1] + [0.06, 0]), "r = resolution", color="tab:blue")
    apothem = 0.5 * seeds[0]
    ax.annotate("", xy=apothem, xytext=(0, 0), arrowprops={"arrowstyle": "<->", "color": "tab:red"})
    ax.text(*(apothem + [0.03, -0.12]), "r/2", color="tab:red")
    ax.set_aspect("equal")
    ax.set_xlim(-1.3 * r, 1.3 * r)
    ax.set_ylim(-1.3 * r, 1.3 * r)
    ax.set_axis_off()
    ax.legend(loc="lower center", fontsize=8, ncol=1, bbox_to_anchor=(0.5, -0.12))
    ax.set_title("hex_ring: six seeds at radius r, cell apothem r/2")
    fig.tight_layout()
    return fig


plot_hex_ring_diagram();

In [ ]:
cm_hex, mg_hex, mesh_s = mesh_case(hex_ring=True)
grid_hex, _, tess_s = tessellate(cm_hex, mg_hex)
print(f"{len(grid_hex)} cells ({len(grid_hex) - len(grid_base)} more than the baseline, "
      f"{(len(grid_hex) - len(grid_base)) / len(WELLS):.0f} per well)")
print(f"Rings intact after meshing: {mg_hex.diagnostics['hex_rings']}")
plot_well_zooms(grid_hex, "hex_ring=True: the well cell is a regular hexagon centred on the well");

In [ ]:
well_hex = well_cell_table(grid_hex)
well_hex["hexagon_area"] = np.sqrt(3.0) / 2.0 * well_hex["resolution"] ** 2
pd.concat({"baseline": well_base, "hex_ring": well_hex}, axis=1).round(3)

The well cells now have zero drift, six neighbours and exactly the area
(sqrt(3)/2) r^2. The neighbour-area CV drops from about 0.17 to about 0.04; the
six ring cells are not identical because their outer sides depend on the
graded mesh beyond the ring. The ring costs about 40 extra cells per well with
`growth_factor=1.2`: the seeds are also targets of the point's size field, so
the refined patch is about one radius larger.

### When the ring is dropped

`ConceptualMesh.generate()` checks each ring after snapping and clipping and
drops it, with a `UserWarning`, in three cases. The point is then meshed as
usual.

- **Clearance:** a polygon or domain boundary, an embedded line (including
  the width of a barrier's straddle pairs or a quad buffer) or another
  embedded point is closer than `HEX_RING_CLEARANCE * resolution`
  (2 x resolution).
- **Size:** a zone, line, point or field-only polygon is estimated to set a
  mesh size below `HEX_RING_MIN_SIZE_RATIO * resolution` (0.9 x resolution)
  at the ring. Gmsh would then put nodes inside the ring and split the
  hexagon. An embedded zone counts at its own resolution throughout its
  interior, so a well inside a zone whose resolution is finer than the
  well's loses its ring.
- **Deduplication:** `simplify_tolerance` merges the point into another one.

The first demo below needs no meshing: one point is 6 m from a river
(clearance 10 m), one sits in a zone with resolution 3 (finer than
0.9 x 5 = 4.5) and one is clear of both.

In [ ]:
cm_drop = ConceptualMesh()
cm_drop.add_polygon(box(0, 0, 300, 200), zone_id=1)
cm_drop.add_polygon(box(180, 40, 280, 160), zone_id="fine", resolution=3, z_order=1)
cm_drop.add_line(LineString([(0, 106), (150, 106)]), line_id="river", resolution=2)
cm_drop.add_point(Point(75, 100), point_id="near-river", resolution=5, hex_ring=True)
cm_drop.add_point(Point(230, 100), point_id="in-fine-zone", resolution=5, hex_ring=True)
cm_drop.add_point(Point(75, 30), point_id="clear", resolution=5, hex_ring=True)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    _, _, clean_points = cm_drop.generate()
for warning in caught:
    print(f"{warning.category.__name__}: {warning.message}")
clean_points.assign(has_ring=clean_points["ring_seeds"].notna())[["point_id", "lc", "hex_ring", "has_ring"]]

The size estimate covers the default growth fields and explicit
`GeometricGrowthField` / `ThresholdField` entries. Anything else (other
explicit fields, or a `background_lc` below the ring radius) is caught after
meshing: `MeshGenerator.generate()` checks that each ring's centre node has
exactly six triangles onto its six seeds, warns when it does not, and records
the result per point in `mg.diagnostics["hex_rings"]`. Here `background_lc`
(3 m) is finer than the ring radius (5 m):

In [ ]:
cm_broken = ConceptualMesh()
cm_broken.add_polygon(box(0, 0, 100, 100), zone_id=1)
cm_broken.add_point(Point(50, 50), point_id="w", resolution=5, hex_ring=True)
clean_broken = cm_broken.generate()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    mg_broken = MeshGenerator(background_lc=3, verbosity=0)
    mg_broken.generate(*clean_broken)
for warning in caught:
    print(f"{warning.category.__name__}: {warning.message}")
print(f"diagnostics['hex_rings']: {mg_broken.diagnostics['hex_rings']}")

| `hex_ring=True` | |
|---|---|
| **Pros** | Exact centring (drift 0) and a symmetric first ring of cells, by construction. |
| | Cheap: six extra nodes per point; meshing time does not change measurably. |
| | Deterministic and local: the rest of the grid is the same kind of mesh as before. |
| **Cons** | About 40 extra cells per point at `growth_factor=1.2`. |
| | Dropped (with a warning) when a polygon boundary, line or other point is closer than 2 x `resolution`. |
| | Dropped (with a warning) where a finer size field reaches the ring, e.g. inside a zone whose resolution is finer than 0.9 x the well's. |
| | Point features only. |
| | The ring radius is tied to `resolution`; the well cell's apothem is `resolution / 2`. |
| | The refined patch grows by about one ring radius, because the size field also targets the seeds. |

## 4. Lloyd relaxation (`lloyd_iterations`)

Lloyd's algorithm moves each generator to the centroid of its Voronoi cell and
repeats. Its fixed point is a centroidal Voronoi tessellation (CVT): every
generator is its cell's centroid. `VoronoiTessellator(mg, cm, lloyd_iterations=N)`
runs N passes on the Gmsh nodes before building the grid.

**Density weighting.** Plain centroids drive a CVT towards equal cell sizes,
which removes the refinement around features. With a density rho(x) the CVT
spacing scales as rho^(-1/4) in 2D, so vorflow weights the centroids with
rho = h^-4, where h is the local mesh size. h is interpolated from
`MeshGenerator.node_sizes` (the mean length of the mesh edges at each node),
because the Gmsh size field cannot be evaluated once Gmsh has shut down. These
sizes jitter by about 10% between neighbouring nodes, i.e. +-30-45% in rho,
and Lloyd would converge to that noise, so they are first smoothed in log
space over the mesh edges (`lloyd_size_smoothing`, 5 passes by default; fixed
nodes keep their size and anchor the grading). Each cell's weighted centroid
is integrated over a fan of triangles from its generator with a 3-point rule
per triangle.

**Fixed and free nodes.** Only interior nodes of the embedded polygon surfaces
move (`MeshGenerator.node_is_free`). Nodes on the domain and zone boundaries,
embedded points and their hex-ring seeds, embedded and barrier lines (and
their straddle pairs) and quad-buffer nodes stay fixed, so every feature keeps
its exact geometry.

**Rejected moves.** A move is rejected for that pass if the new position leaves
the node's zone (minus any quad-buffer footprint) or the segment from old to
new position crosses an embedded line. The node then stays where it is.

In [ ]:
def plot_free_nodes(mg: MeshGenerator, focus: Point, half_width: float, domain=DOMAIN, zone=UPPER_ZONE):
    """Free (moved by Lloyd) and fixed mesh nodes, whole domain and around ``focus``."""
    free = mg.node_is_free
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    for ax, size in zip(axes, (3, 8)):
        ax.plot(*domain.exterior.xy, color="k", lw=1)
        ax.plot(*zone.exterior.xy, color="tab:blue", lw=1)
        ax.plot(mg.nodes[free, 0], mg.nodes[free, 1], ".", color="0.6", ms=size, label="free")
        ax.plot(mg.nodes[~free, 0], mg.nodes[~free, 1], "o", color="tab:red", ms=size / 2, label="fixed")
        ax.set_aspect("equal")
    axes[1].set_xlim(focus.x - half_width, focus.x + half_width)
    axes[1].set_ylim(focus.y - half_width, focus.y + half_width)
    axes[0].set_title(f"{free.sum()} free, {(~free).sum()} fixed nodes")
    axes[1].set_title("Around w0 (hex_ring=True): the point and its six seeds are fixed")
    axes[0].legend(loc="lower right")
    fig.tight_layout()
    return fig


plot_free_nodes(mg_hex, WELLS["w0"][0], half_width=30.0);

### Convergence

The same mesh is tessellated with increasing `lloyd_iterations`.
`tessellator.lloyd_report["max_rel_shift"]` is the residual of the last pass:
the largest remaining distance from a node to its weighted centroid,
|c - x| / h, over the nodes whose move was accepted. It does not depend on
`lloyd_damping`, and `lloyd_tolerance` is compared against it. Run with
increasing `lloyd_iterations`, it traces the convergence history.

In [ ]:
LLOYD_STEPS = [0, 1, 2, 5, 10, 20, 40, 100]
rows, lloyd_grids = [], {}
for k in LLOYD_STEPS:
    grid, tess, tess_s = tessellate(cm_base, mg_base, lloyd_iterations=k)
    lloyd_grids[k] = grid
    report = tess.lloyd_report or {"max_rel_shift": np.nan, "rejected": 0, "n_free": 0}
    rows.append({
        **summarise(f"lloyd {k}", grid, tess_s),
        "iterations": k,
        "max_rel_shift": report["max_rel_shift"],
        "rejected": report["rejected"],
    })
convergence = pd.DataFrame(rows).set_index("iterations")
convergence[["runtime_s", "max_rel_shift", "rejected", "int_drift_p95", "ortho_p95_centroid", "well_drift", "well_cell_area", "short_face_cells"]]

In [ ]:
def plot_convergence(table: pd.DataFrame):
    """Residual max_rel_shift, interior p95 drift and p95 centroid ortho_error against the number of passes."""
    relaxed = table.loc[table.index > 0]
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
    axes[0].semilogy(relaxed.index, relaxed["max_rel_shift"], "o-")
    axes[0].axhline(1e-3, color="0.5", ls="--", lw=0.8, label="default lloyd_tolerance")
    axes[0].set_title("Residual max |c - x| / h, last pass")
    axes[0].legend(fontsize=8)
    axes[1].plot(table.index, table["int_drift_p95"], "o-")
    axes[1].set_title("Interior drift_ratio, p95")
    axes[2].plot(table.index, table["ortho_p95_centroid"], "o-")
    axes[2].set_title("Centroid ortho_error p95 (degrees)")
    for ax in axes:
        ax.set_xlabel("lloyd_iterations")
    fig.tight_layout()
    return fig


plot_convergence(convergence);

Most of the gain in drift and orthogonality comes in the first five to ten
passes. The last column is the share of interior cells with an edge shorter
than 0.1 x sqrt(cell area). There are none without Lloyd. The count rises
while the grid is partly relaxed, peaks around 20 passes (a partly relaxed
grid passes through near-cocircular generators) and falls again by 100.
Use 100 or more passes, not about 20. On a graded mesh the residual stays
above the default `lloyd_tolerance` of 1e-3, so the run usually uses every
pass it is given. `rejected` counts moves that would have left a node's zone
or crossed an embedded line; none were rejected here.

In [ ]:
def interior_quality(grid, domain=DOMAIN):
    """calculate_mesh_quality of the cells that do not touch the domain boundary."""
    quality = calculate_mesh_quality(grid)
    return quality[~quality.geometry.intersects(domain.boundary)]


def plot_histograms(grids: dict):
    """Interior drift_ratio and centroid ortho_error histograms for each named grid."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    drift_bins = np.linspace(0.0, 0.25, 51)
    ortho_bins = np.linspace(0.0, 12.0, 49)
    for name, grid in grids.items():
        drift = interior_quality(grid)["drift_ratio"]
        ortho = build_connectivity(grid, center="centroid")["ortho_error"]
        axes[0].hist(drift, bins=drift_bins, histtype="step", lw=1.5, label=name)
        axes[1].hist(ortho, bins=ortho_bins, histtype="step", lw=1.5, label=name)
    axes[0].set_xlabel("drift_ratio (interior cells)")
    axes[1].set_xlabel("ortho_error, centroid connectivity (degrees)")
    axes[1].set_yscale("log")
    for ax in axes:
        ax.set_ylabel("count")
        ax.legend()
    fig.tight_layout()
    return fig


plot_histograms({"defaults": lloyd_grids[0], "lloyd_iterations=100": lloyd_grids[100]});

Lloyd shifts the bulk of both distributions down. The `ortho_error` tail above
about 6 degrees does not change: almost all of those connections involve a
cell on the domain boundary, whose generator is a fixed boundary node.
`VoronoiTessellator(boundary_centering="inset_mirror")` targets those cells and
can be combined with `lloyd_iterations`.

In [ ]:
def plot_lloyd_shift(grid, domain=DOMAIN, zone=UPPER_ZONE, wells=WELLS):
    """Cells coloured by how far their generator moved, relative to the cell size."""
    relative = grid["lloyd_shift"] / np.sqrt(grid.geometry.area)
    fig, ax = plt.subplots(figsize=(7.5, 7))
    grid.assign(relative_shift=relative).plot(
        ax=ax, column="relative_shift", cmap="viridis", legend=True, vmin=0, vmax=0.3,
        edgecolor="0.5", linewidth=0.1, legend_kwds={"label": "lloyd_shift / sqrt(cell area)", "shrink": 0.8},
    )
    ax.plot(*zone.exterior.xy, color="w", lw=0.8)
    for point, _ in wells.values():
        ax.plot(point.x, point.y, "r*", ms=8)
    ax.plot(*domain.exterior.xy, color="k", lw=1)
    ax.set_aspect("equal")
    ax.set_title("Generator moves after 100 passes (0 for fixed nodes)")
    fig.tight_layout()
    return fig


plot_lloyd_shift(lloyd_grids[100]);

Moves of 10-30% of the cell size are scattered through the graded transitions
around the wells and the coarse interior. Cells of fixed nodes (domain
boundary, zone edge, wells) have `lloyd_shift` 0.

### Grading is kept

To check that the refinement survives, the next cell compares the cell size
around w0 (resolution 5 m) for the defaults, the weighted Lloyd run and an
*unweighted* Lloyd run. The unweighted run is not an option in vorflow; it is
made here by giving a copy of the mesh generator a constant `node_sizes`, which
makes rho uniform. Cell size is shown as the spacing of a regular hexagonal
grid with the same cell area, sqrt(2 A / sqrt(3)), which equals the mesh size h
for a regular mesh; the dashed line is the size field target
h(d) = 5 + 0.2 d, capped at `background_lc`.

In [ ]:
mg_flat = copy.copy(mg_base)
mg_flat.node_sizes = np.full(len(mg_base.nodes), BACKGROUND_LC)
grid_flat, _, _ = tessellate(cm_base, mg_flat, lloyd_iterations=100)


def size_profile(grid, point: Point, max_distance: float, n_bins: int = 25) -> pd.Series:
    """Median equivalent hexagon spacing sqrt(2 A / sqrt(3)) of cells binned by generator distance from ``point``."""
    distance = np.hypot(grid["x"] - point.x, grid["y"] - point.y)
    spacing = np.sqrt(2.0 * grid.geometry.area / np.sqrt(3.0))
    bins = np.linspace(0.0, max_distance, n_bins + 1)
    binned = pd.cut(distance, bins)
    profile = spacing.groupby(binned, observed=True).median()
    profile.index = [interval.mid for interval in profile.index]
    return profile


def plot_size_profiles(grids: dict, well_id: str, wells=WELLS, max_distance: float = 300.0,
                       growth_factor=GROWTH_FACTOR, background_lc=BACKGROUND_LC):
    """Cell size against distance from a well for each named grid, with the size-field target."""
    point, resolution = wells[well_id]
    fig, ax = plt.subplots(figsize=(7, 4))
    for name, grid in grids.items():
        profile = size_profile(grid, point, max_distance)
        ax.plot(profile.index, profile.values, "o-", ms=3, label=name)
    d = np.linspace(0.0, max_distance, 100)
    ax.plot(d, np.minimum(resolution + (growth_factor - 1.0) * d, background_lc), "k--", lw=1, label="target h(d)")
    ax.set_xlabel(f"distance from {well_id} (m)")
    ax.set_ylabel("equivalent hexagon spacing (m)")
    ax.legend()
    fig.tight_layout()
    return fig


grading_grids = {"defaults": grid_base, "weighted Lloyd 100": lloyd_grids[100], "unweighted Lloyd 100": grid_flat}
plot_size_profiles(grading_grids, "w0");
pd.DataFrame({name: well_cell_table(grid)["cell_area"] for name, grid in grading_grids.items()})

The weighted run follows the default grading. Without `hex_ring` the well
cells grow by 5-60% (w0 34 -> 39 m², w2 29 -> 46 m²). Gmsh makes the first
ring of nodes around a well 5-15% finer than the size field asks for, and the
smoothed density moves that ring to about the requested size. `hex_ring` pins
the well cells (section 5). The unweighted run evens out cell sizes: the
spacing next to w0 rises from about 7 to 26 m, the cells 200-300 m out get
finer, and the well cells grow 15-30 times in area. This is why vorflow
weights the centroids.

| `lloyd_iterations` | |
|---|---|
| **Pros** | Better centroidality across the whole grid: interior p95 `drift_ratio` 0.118 -> 0.070 (100 passes). |
| | Better MODFLOW-facing (centroid) orthogonality: p95 `ortho_error` 4.1 -> 1.4 degrees. |
| | Keeps the mesh grading (density h^-4 from smoothed node sizes). |
| | Opt-in, no new dependencies (NumPy, SciPy, Shapely). |
| **Cons** | Extra runtime: each pass is one Voronoi diagram plus weighted centroids; 100 passes cost several times as much as meshing and tessellating once. |
| | Adds a few very short cell faces (none without Lloyd), most around 20 passes. |
| | The grid is no longer the exact dual of the triangle mesh (`MeshGenerator.get_element_grid()`). |
| | Cells next to fixed nodes (domain and zone edges, lines, wells) improve little. |
| | The well's own cell is not exactly centred (the well node is fixed), and without `hex_ring` it grows towards the size-field target. |
| | Graded meshes do not reach the default `lloyd_tolerance`; the run uses all passes. |
| | Zone edges and embedded lines can reject moves, which leaves those nodes unrelaxed for that pass. |

## 5. Combining both, and recommendations

`hex_ring` fixes the well cells exactly; Lloyd improves the rest of the grid.
The ring seeds are fixed nodes, so Lloyd leaves the hexagons untouched and the
two compose. The table reruns every case (meshing plus tessellation runtime).

In [ ]:
SUMMARY_CASES = [
    # (name, hex_ring, MeshGenerator kwargs, lloyd_iterations)
    ("baseline defaults", False, {}, 0),
    ("smoothing 0/0", False, {"smoothing_steps": 0, "optimization_cycles": 0}, 0),
    ("smoothing 100/10", False, {"smoothing_steps": 100, "optimization_cycles": 10}, 0),
    ("hex_ring", True, {}, 0),
    ("lloyd 20", False, {}, 20),
    ("lloyd 100", False, {}, 100),
    ("hex_ring + lloyd 100", True, {}, 100),
]
rows, summary_grids = [], {}
for name, hex_ring, mg_kwargs, lloyd_iterations in SUMMARY_CASES:
    cm, mg, mesh_s = mesh_case(hex_ring=hex_ring, mg_kwargs=mg_kwargs)
    grid, _, tess_s = tessellate(cm, mg, lloyd_iterations=lloyd_iterations)
    summary_grids[name] = grid
    rows.append(summarise(name, grid, mesh_s + tess_s))
summary = pd.DataFrame(rows).set_index("case")
summary.round(3)

In [ ]:
plot_well_zooms(summary_grids["hex_ring + lloyd 100"], "hex_ring=True with lloyd_iterations=100");

Recommendations:

- Use `hex_ring=True` when the cell around a well or observation point must be
  centred and symmetric, e.g. when the well-bore head or near-well drawdown is
  computed from that cell (Peaceman-type well corrections assume a regular
  cell around the well). Keep other features at least 2 x `resolution` away,
  or the ring is dropped.
- Use `lloyd_iterations` when the model uses cell centroids as cell centres,
  or for visually more regular cells. Use 100 or more passes with
  `hex_ring=True`, and avoid about 20 passes, where the short-face count peaks.
  Skip Lloyd if you need the Voronoi grid to stay the exact dual of
  `get_element_grid()`.
- Head accuracy depends mainly on `growth_factor`, not on cell shape. In a
  steady radial-flow test with generator centres
  ([#31](https://github.com/rhugman/vorflow/issues/31)), halving
  `growth_factor - 1` roughly doubled the cell count and cut the head RMSE
  about threefold. 100 Lloyd passes improved it by at most about 5%. Lower
  `growth_factor` (or the resolution) where accuracy matters.
- The two compose: `hex_ring` for the points, Lloyd for everything else.
  Domain-boundary cells are handled by `boundary_centering="inset_mirror"`.
- Neither changes the defaults; existing models produce the same grid.